In [ ]:
pip install trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 13.1 MB/s eta 0:00:00


For urls pure only

In [ ]:
import trafilatura

url_1 = "https://www.aera.net/Newsroom/Trending-Topic-Research-Files/Trending-Topic-Research-File-Education-Technology-and-Online-Learning"
url_2= "https://elearningindustry.com/the-use-of-technology-in-online-education"
url_3= "https://education.purdue.edu/news/2024/01/01/how-has-technology-changed-education/"
url_4="https://pmc.ncbi.nlm.nih.gov/articles/PMC9247945/"
html1= trafilatura.fetch_url(url_1)
html2 = trafilatura.fetch_url(url_2)
html3= trafilatura.fetch_url(url_3)
html4= trafilatura.fetch_url(url_4)
print(type(html1))
print(html1[:500])
print(type(html2))
print(html2[:500])
print(type(html3))
print(html3[:500])
print(type(html4))
print(html4[:500])

<class 'str'>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
"http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html  xml:lang="en-US" lang="en-US" xmlns="http://www.w3.org/1999/xhtml">
<head id="Head"><meta content="text/html; charset=UTF-8" http-equiv="Content-Type" />
<meta name="REVISIT-AFTER" content="1 DAYS" />
<meta name="RATING" content="GENERAL" />
<meta name="RESOURCE-TYPE" content="DOCUMENT" />
<meta content="text/javascript" http-equiv="Content-Script-Type" />
<meta con
<class 'str'>
<!DOCTYPE html>

<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en-US"
      lang="en-US">

<head>

    <meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>
<meta http-equiv="Content-Language" content="en-US>"/>

<meta name="Author" content="Rebecca Paine"/>
<meta name="Owner" content="Rebecca Paine"/>
<meta name="Publisher" content="eLearning Industry"/>
<meta name="Copyright" content="Rebecca Paine"/>

<meta name="google-site-verification"
      

Extract the main text

In [ ]:
text1 = trafilatura.extract(html1)
text2 = trafilatura.extract(html2)
text3 = trafilatura.extract(html3)
text4 = trafilatura.extract(html4)

In [ ]:
num1=len(text1.split())
num2=len(text2.split())
num3=len(text3.split())
num4=len(text4.split())
print("word count for first article:",num1)
print("word count for second article:",num2)
print("word count for third article:",num3)
print("word count for fourth article:",num4)

word count for first article: 5055
word count for second article: 815
word count for third article: 735
word count for fourth article: 7105


1.2 Pipeline extraction

In [ ]:
import json

In [ ]:
urls = [
    "https://www.aera.net/Newsroom/Trending-Topic-Research-Files/Trending-Topic-Research-File-Education-Technology-and-Online-Learning",
    "https://elearningindustry.com/the-use-of-technology-in-online-education",
    "https://education.purdue.edu/news/2024/01/01/how-has-technology-changed-education/",
    "https://pmc.ncbi.nlm.nih.gov/articles/PMC9247945/"
]

Useful page

In [ ]:
def is_useful(text, min_words=500):
    if text is None:
        return False
    return len(text.split()) >= min_words

We extract functions

In [ ]:
def extract_main_text(url):
    html = trafilatura.fetch_url(url)
    if html is None:
        return None

    text = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=False
    )
    return text


We run the pipeline and store JSON

In [ ]:
output_file = "edtech_corpus.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for url in urls:
        text = extract_main_text(url)

        if is_useful(text):
            record = {
                "url": url,
                "word_count": len(text.split()),
                "text": text
            }
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
            print(f"Saved: {url}")
        else:
            print(f"Discarded (too short or failed): {url}")


Saved: https://www.aera.net/Newsroom/Trending-Topic-Research-Files/Trending-Topic-Research-File-Education-Technology-and-Online-Learning
Saved: https://elearningindustry.com/the-use-of-technology-in-online-education
Saved: https://education.purdue.edu/news/2024/01/01/how-has-technology-changed-education/
Saved: https://pmc.ncbi.nlm.nih.gov/articles/PMC9247945/


In [ ]:
print(f"Word count: {len(text1.split())}")

Word count: 5055


we load our corpus

In [ ]:
documents = []

with open("edtech_corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(len(documents))
print(documents[0].keys())

4
dict_keys(['url', 'word_count', 'text'])


In [ ]:
pip install spacy

2.1) Named Entity Recognition (NER) with spaCy

This identifies our graph nodes (PERSON, ORG, GPE, DATE).


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully")

spaCy model loaded successfully


Then we extract the labels (nodes)

We only keep useful entity types (as required).

In [ ]:
ALLOWED_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

def extract_entities(text):
    doc = nlp(text)
    entities = []

    for ent in doc.ents:
        if ent.label_ in ALLOWED_LABELS:
            entities.append({
                "text": ent.text,
                "label": ent.label_
            })
    return entities

Let us test it

In [ ]:
entities_sample = extract_entities(documents[0]["text"])
entities_sample[:10]

[{'text': 'Trending Topic Research File\nEducators', 'label': 'ORG'},
 {'text': 'AERA', 'label': 'ORG'},
 {'text': '2002', 'label': 'DATE'},
 {'text': 'Limitations', 'label': 'ORG'},
 {'text': 'District Digital Capacity for Education', 'label': 'ORG'},
 {'text': 'Emily E. N. Miller', 'label': 'PERSON'},
 {'text': 'Sarah Pedersen AERA Open', 'label': 'PERSON'},
 {'text': 'December 2024', 'label': 'DATE'},
 {'text': 'nonmetropolitan', 'label': 'GPE'},
 {'text': 'Louisiana', 'label': 'GPE'}]

The Named Entity Recognition step successfully identified relevant entities such as organizations (AERA), persons (Emily E. N. Miller), dates (December 2024), and locations (Louisiana). However, some extracted entities were overly generic or incorrectly classified, such as section headers or common nouns. This highlights the need for a refinement phase to filter noise and retain only domain relevant entities, which is a common challenge when applying general-purpose NER models to academic and web based texts.

2.2 Introduction to Relations

(We identify the “Edges” of the Knowledge Graph)

In [ ]:
def extract_relations(text):
    doc = nlp(text)
    relations = []

    for sent in doc.sents:
        ents = [ent for ent in sent.ents if ent.label_ in ALLOWED_LABELS]

        if len(ents) >= 2:
            for i in range(len(ents) - 1):
                relations.append({
                    "source": ents[i].text,
                    "target": ents[i + 1].text,
                    "sentence": sent.text
                })

    return relations

In [ ]:
def extract_verb_relations(text):

    doc = nlp(text)
    triples = []

    for sent in doc.sents:
        subject = None
        verb = None
        obj = None

        for token in sent:
            if token.dep_ == "nsubj":
                subject = token.text
                verb = token.head.text
            elif token.dep_ == "dobj":
                obj = token.text

        if subject and verb and obj:
            triples.append({
                "subject": subject,
                "relation": verb,
                "object": obj,
                "sentence": sent.text
            })

    return triples

In [ ]:
sample_text = documents[0]["text"]

relations = extract_relations(sample_text)

# Show first 5 relations
for r in relations[:5]:
    print("SOURCE:", r["source"])
    print("TARGET:", r["target"])
    print("SENTENCE:", r["sentence"])
    print("-" * 50)

SOURCE: AERA
TARGET: 2002
SENTENCE: The following compendium of open-access articles are inclusive of all substantive AERA journal content regarding education technology and online learning published since 2002.
--------------------------------------------------
SOURCE: Limitations
TARGET: District Digital Capacity for Education
SENTENCE: Promises and Limitations in District Digital Capacity for Education During COVID-19 Emily E. N. Miller, Sarah Pedersen AERA Open, December 2024
--------------------------------------------------
SOURCE: District Digital Capacity for Education
TARGET: Emily E. N. Miller
SENTENCE: Promises and Limitations in District Digital Capacity for Education During COVID-19 Emily E. N. Miller, Sarah Pedersen AERA Open, December 2024
--------------------------------------------------
SOURCE: Emily E. N. Miller
TARGET: Sarah Pedersen AERA Open
SENTENCE: Promises and Limitations in District Digital Capacity for Education During COVID-19 Emily E. N. Miller, Sarah Pede

In [ ]:
verb_relations = extract_verb_relations(sample_text)

# Show first 5 verb-based triples
for vr in verb_relations[:5]:
    print("SUBJECT:", vr["subject"])
    print("RELATION:", vr["relation"])
    print("OBJECT:", vr["object"])
    print("SENTENCE:", vr["sentence"])
    print("-" * 50)

SUBJECT: areas
RELATION: experience
OBJECT: benefits
SENTENCE: However, nonmetropolitan and Appalachian areas did not experience the same benefits from their digital capacity investments compared with their metropolitan counterparts.

--------------------------------------------------
SUBJECT: study
RELATION: examines
OBJECT: impact
SENTENCE: This study examines the impact of an online math learning program on third through fifth grade math achievement in Louisiana.

--------------------------------------------------
SUBJECT: researchers
RELATION: reflect
OBJECT: research
SENTENCE: Based on experiences during the global health pandemic, researchers reflect on these limitations and offer thoughts to motivate a new vision of family learning research.

--------------------------------------------------
SUBJECT: Researchers
RELATION: found
OBJECT: information
SENTENCE: Researchers found significant growth in students’ ability to evaluate the credibility of online content with materials des

Let us compare those two:

In [ ]:
print("Sentence-level relations:", len(relations))
print("Verb-based relations:", len(verb_relations))

Sentence-level relations: 410
Verb-based relations: 85


The sentence level approach produces a high number of relations, as it connects entities that co-occur within the same sentence. This method maximizes recall but may include weak or implicit relations.